# Quick start

This notebook walks through the smallest useful drift check you can do with `drift-control`:
compare a reference feature distribution against a current one and find out whether anything has shifted.

If you can run this top-to-bottom you have a working install.

In [ ]:
import numpy as np

from drift_control.unified_drift_detector import UnifiedDriftDetector

rng = np.random.default_rng(0)

## 1. Two distributions, one detector

We make a *reference* sample (your training data) and a *current* sample (what's flowing through
production today). The Kolmogorov–Smirnov test is the cheapest sanity check for continuous features.

In [ ]:
reference = rng.normal(loc=0.0, scale=1.0, size=500)
current = rng.normal(loc=0.7, scale=1.0, size=500)

ks = UnifiedDriftDetector(method="ks", alpha=0.05)
out = ks.detect_drift(reference, current)
print(f"KS drift={out.drift!r}  p_value={out.p_value:.4f}")

## 2. PSI for batch pipelines

PSI is what most production ML teams already log because it has well-known thresholds
(rule of thumb: `<0.1` stable, `0.1–0.25` watch, `>0.25` shifted). We bin against the reference and read off the score.

In [ ]:
psi = UnifiedDriftDetector(method="psi", threshold=0.2, bins=10, strategy="quantile")
out = psi.detect_drift(reference, current)
print(f"PSI score={out.score:.4f}  drift={out.drift!r}")

## 3. The unified interface

`UnifiedDriftDetector` is a thin facade over every univariate detector in the package.
It returns a `DriftResult` dataclass with the same shape regardless of method,
which makes ensembling and logging much easier downstream.

In [ ]:
for method in ("psi", "ks", "cvm", "js", "wasserstein"):
    detector = UnifiedDriftDetector(method=method)
    out = detector.detect_drift(reference, current)
    print(
        f"{method:>12}  drift={str(out.drift):<5}  score={out.score:.4f}  "
        f"p_value={'n/a' if out.p_value is None else f'{out.p_value:.4f}'}"
    )

## 4. No drift case

Sanity check — the same detector should *not* fire when the two samples are drawn from the same distribution.

In [ ]:
ref_clean = rng.normal(0, 1, size=500)
cur_clean = rng.normal(0, 1, size=500)

out = UnifiedDriftDetector(method="ks").detect_drift(ref_clean, cur_clean)
print(f"KS on no-drift sample: drift={out.drift!r}  p_value={out.p_value:.4f}")

## Where to go next

- `02_detector_selection.ipynb` — which detector catches which kind of shift, with synthetic scenarios.
- `03_streaming.ipynb` — `StreamMonitor` for production batch streams, with sliding-window re-baselining and drift callbacks.
- `04_ensemble_and_multiple_testing.ipynb` — voting across detectors and Benjamini–Hochberg correction when scoring many columns at once.
- The `drift-control` CLI (`drift-control --help`) wraps everything here for CI/CD pipelines.